## 1. Устанавливаю спарк на 1 же окружение

In [33]:
import pyspark 
from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext, HiveContext
from pyspark.sql import functions as F 
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [34]:
# проверяю какое окружение стоит
import sys
print("Jupyter python:", sys.executable)

Jupyter python: /Users/user/Full_ML_Course/.venv/bin/python


In [35]:
# привязываю спарк к этому же окружению
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])
print("PYSPARK_DRIVER_PYTHON:", os.environ["PYSPARK_DRIVER_PYTHON"])

PYSPARK_PYTHON: /Users/user/Full_ML_Course/.venv/bin/python
PYSPARK_DRIVER_PYTHON: /Users/user/Full_ML_Course/.venv/bin/python


In [36]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("check-python").getOrCreate()

print("spark.pyspark.python =", spark.sparkContext.getConf().get("spark.pyspark.python"))
print("spark.pyspark.driver.python =", spark.sparkContext.getConf().get("spark.pyspark.driver.python"))

spark.pyspark.python = None
spark.pyspark.driver.python = None


In [37]:
r = spark.sparkContext.parallelize([1], 1).map(lambda _: __import__("sys").executable).collect()
print("Executor python:", r[0])

Executor python: /Users/user/Full_ML_Course/.venv/bin/python


In [38]:
df = spark.createDataFrame(
    [("Betty_White", 288886), ("Main_Page", 139564)],
    ["article_title", "view_count"]
)
df.show()

+-------------+----------+
|article_title|view_count|
+-------------+----------+
|  Betty_White|    288886|
|    Main_Page|    139564|
+-------------+----------+



## 2. Работа на спарке

In [39]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate() 

### Spark SQL

In [40]:
# подключение к спарку

spark = SparkSession.builder\
    .config('spark.app.name', 'learning_spark_sql')\
    .getOrCreate()
    
print(spark.sparkContext) 
# <SparkContext master=local[*] appName=learning_spark_sql>

<SparkContext master=local[*] appName=check-python>


26/02/25 10:55:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [41]:
# Create an RDD from a list
hrly_views_rdd  = spark.sparkContext.parallelize([
    ["Betty_White" , 288886],
    ["Main_Page", 139564],
    ["New_Year's_Day", 7892],
    ["ABBA", 8154]
])

# Convert RDD to DataFrame
hrly_views_df = hrly_views_rdd.toDF(["article_title", "view_count"])
hrly_views_df.show(4, truncate=False)

+--------------+----------+
|article_title |view_count|
+--------------+----------+
|Betty_White   |288886    |
|Main_Page     |139564    |
|New_Year's_Day|7892      |
|ABBA          |8154      |
+--------------+----------+



In [42]:
# Доступ к RDD, лежащему в основе DataFrame
hrly_views_df_rdd = hrly_views_df.rdd

# Проверьте тип объекта
print(type(hrly_views_df_rdd)) 
# <класс 'pyspark.rdd.RDD'>

<class 'pyspark.core.rdd.RDD'>


In [43]:
print(type(spark.read)) 
# <class 'pyspark.sql.readwriter.DataFrameReader'>

# Read CSV to DataFrame
hrly_views_df = spark.read\
.option('header', True) \
.option('delimiter', ',') \
.option('inferSchema', True)  \
.csv('/Users/user/Full_ML_Course/08-Linear-Regression-Models/Advertising.csv')

# Display first 5 rows of DataFrame
hrly_views_df.show(5, truncate=False)

<class 'pyspark.sql.readwriter.DataFrameReader'>
+-----+-----+---------+-----+
|TV   |radio|newspaper|sales|
+-----+-----+---------+-----+
|230.1|37.8 |69.2     |22.1 |
|44.5 |39.3 |45.1     |10.4 |
|17.2 |45.9 |69.3     |9.3  |
|151.5|41.3 |58.5     |18.5 |
|180.8|10.8 |58.4     |12.9 |
+-----+-----+---------+-----+
only showing top 5 rows


В этом коде есть несколько вещей, давайте рассмотрим их по очереди:

Этот код использует функцию SparkSession.read для создания нового DataFrameReader

У DataFrameReader есть метод .option('option_name', 'option_value'), который можно использовать, чтобы указать Spark, как именно читать файл. В данном случае мы использовали следующие опции:

.option('header', True) - Указывает, что файл уже содержит строку заголовка. По умолчанию Spark считает, что заголовка нет.

.option('delimiter', ' ') - Указывает, что каждый столбец разделен пробелом (' '). По умолчанию Spark считает, что столбцы CSV разделяются запятыми.

.option('inferSchema', True) - указывает Spark на выборку подмножества строк перед определением типа каждого столбца. По умолчанию Spark будет рассматривать все столбцы CSV как строки.

У DataFrameReader также есть метод .csv('path'), который загружает CSV-файл и возвращает результат в виде DataFrame. Есть несколько быстрых способов проверить, что наши данные были считаны правильно. Самый прямой способ - это проверка DataFrame.show().

In [44]:
hrly_views_df.describe().show()

+-------+-----------------+------------------+------------------+------------------+
|summary|               TV|             radio|         newspaper|             sales|
+-------+-----------------+------------------+------------------+------------------+
|  count|              200|               200|               200|               200|
|   mean|         147.0425|23.264000000000024|30.553999999999995|14.022500000000003|
| stddev|85.85423631490805|14.846809176168728| 21.77862083852283| 5.217456565710477|
|    min|              0.7|               0.0|               0.3|               1.6|
|    max|            296.4|              49.6|             114.0|              27.0|
+-------+-----------------+------------------+------------------+------------------+



In [45]:
hrly_views_df.drop('sales').show(5)

+-----+-----+---------+
|   TV|radio|newspaper|
+-----+-----+---------+
|230.1| 37.8|     69.2|
| 44.5| 39.3|     45.1|
| 17.2| 45.9|     69.3|
|151.5| 41.3|     58.5|
|180.8| 10.8|     58.4|
+-----+-----+---------+
only showing top 5 rows


In [46]:
hrly_views_df.withColumnRenamed('radio','Radio').withColumnRenamed('newspaper','Newspaper').show(5)

+-----+-----+---------+-----+
|   TV|Radio|Newspaper|sales|
+-----+-----+---------+-----+
|230.1| 37.8|     69.2| 22.1|
| 44.5| 39.3|     45.1| 10.4|
| 17.2| 45.9|     69.3|  9.3|
|151.5| 41.3|     58.5| 18.5|
|180.8| 10.8|     58.4| 12.9|
+-----+-----+---------+-----+
only showing top 5 rows


## 3. Запросы SQL к Pyspark Dataframes

In [47]:
df = hrly_views_df
df.show(5)

+-----+-----+---------+-----+
|   TV|radio|newspaper|sales|
+-----+-----+---------+-----+
|230.1| 37.8|     69.2| 22.1|
| 44.5| 39.3|     45.1| 10.4|
| 17.2| 45.9|     69.3|  9.3|
|151.5| 41.3|     58.5| 18.5|
|180.8| 10.8|     58.4| 12.9|
+-----+-----+---------+-----+
only showing top 5 rows


In [48]:
# filter аналогичен where 
df.filter(df.TV > 230).show(5, truncate=False)

+-----+-----+---------+-----+
|TV   |radio|newspaper|sales|
+-----+-----+---------+-----+
|230.1|37.8 |69.2     |22.1 |
|281.4|39.6 |55.8     |24.4 |
|237.4|5.1  |23.5     |12.5 |
|262.9|3.5  |19.5     |12.0 |
|240.1|16.7 |22.9     |15.9 |
+-----+-----+---------+-----+
only showing top 5 rows


In [49]:
df.filter(df.TV > 230).select('TV','radio','newspaper').orderBy('TV', ascending=False).show(5)

+-----+-----+---------+
|   TV|radio|newspaper|
+-----+-----+---------+
|296.4| 36.3|    100.9|
|293.6| 27.7|      1.8|
|292.9| 28.3|     43.2|
|290.7|  4.1|      8.5|
|289.7| 42.3|     51.2|
+-----+-----+---------+
only showing top 5 rows


In [50]:
from pyspark.sql import functions as F

df = df.withColumn("rand_col",(F.rand() * 2).cast("int") + 1)
df.show(5)

+-----+-----+---------+-----+--------+
|   TV|radio|newspaper|sales|rand_col|
+-----+-----+---------+-----+--------+
|230.1| 37.8|     69.2| 22.1|       1|
| 44.5| 39.3|     45.1| 10.4|       2|
| 17.2| 45.9|     69.3|  9.3|       1|
|151.5| 41.3|     58.5| 18.5|       1|
|180.8| 10.8|     58.4| 12.9|       2|
+-----+-----+---------+-----+--------+
only showing top 5 rows


### Запрос к PySpark с помощью SQL

In [51]:
# Следующий код сохраняет DataFrame как локальное временное представление в памяти. Пока активна текущая SparkSession, мы можем использовать SparkSession.sql() для запроса к нему.

df.createOrReplaceTempView('advertising')

In [52]:
query = """
    SELECT * 
    FROM advertising 
    WHERE TV > 230
"""
spark.sql(query).show(5, truncate=False)

+-----+-----+---------+-----+--------+
|TV   |radio|newspaper|sales|rand_col|
+-----+-----+---------+-----+--------+
|230.1|37.8 |69.2     |22.1 |1       |
|281.4|39.6 |55.8     |24.4 |2       |
|237.4|5.1  |23.5     |12.5 |2       |
|262.9|3.5  |19.5     |12.0 |2       |
|240.1|16.7 |22.9     |15.9 |2       |
+-----+-----+---------+-----+--------+
only showing top 5 rows


In [53]:
query = """
    SELECT TV, radio, newspaper, rand_col
    FROM advertising
    WHERE TV > 230
    ORDER BY TV DESC
    """

spark.sql(query).show(5, truncate=False)

+-----+-----+---------+--------+
|TV   |radio|newspaper|rand_col|
+-----+-----+---------+--------+
|296.4|36.3 |100.9    |1       |
|293.6|27.7 |1.8      |2       |
|292.9|28.3 |43.2     |2       |
|290.7|4.1  |8.5      |1       |
|289.7|42.3 |51.2     |1       |
+-----+-----+---------+--------+
only showing top 5 rows


In [54]:
query = """
    SELECT rand_col, SUM(TV) as sum_TV
    FROM advertising
    GROUP BY rand_col
    ORDER BY sum_TV DESC
    """

spark.sql(query).show(5, truncate=False)

+--------+------------------+
|rand_col|sum_TV            |
+--------+------------------+
|2       |16254.300000000001|
|1       |13154.200000000004|
+--------+------------------+



In [ ]:
# Подобно методу SparkSession.read(), Spark предлагает метод SparkSession.write(). 
# Давайте немного изменим наш исходный набор данных Wikipedia views и сохраним его на диск. 
# Этот код просто использует .select() для выбора всех столбцов, кроме столбца monthly_count (напомним, что ранее мы обнаружили, что этот столбец содержит только нули).

# Поскольку Spark выполняет все операции параллельно, типично записывать DataFrames в каталог файлов, а не в один CSV-файл. 
# В примере ниже Spark разделит базовый набор данных и запишет несколько CSV-файлов в каталог clean/csv/views_2022_01_01_000000/. 
# Мы также можем использовать аргумент mode метода .csv(), чтобы перезаписать все существующие данные в целевом каталоге.

In [60]:
query = """
    SELECT * 
    FROM advertising 
"""
df = spark.sql(query)
df.show(5)

+-----+-----+---------+-----+--------+
|   TV|radio|newspaper|sales|rand_col|
+-----+-----+---------+-----+--------+
|230.1| 37.8|     69.2| 22.1|       1|
| 44.5| 39.3|     45.1| 10.4|       2|
| 17.2| 45.9|     69.3|  9.3|       1|
|151.5| 41.3|     58.5| 18.5|       1|
|180.8| 10.8|     58.4| 12.9|       2|
+-----+-----+---------+-----+--------+
only showing top 5 rows


In [64]:
# Поскольку Spark выполняет все операции параллельно, типично записывать DataFrames в каталог файлов, а не в один CSV-файл. 
# В примере ниже Spark разделит базовый набор данных и запишет несколько CSV-файлов в каталог clean/csv/views_2022_01_01_000000/. 
# Мы также можем использовать аргумент mode метода .csv(), чтобы перезаписать все существующие данные в целевом каталоге.

# сохранение датафрейма
df.select('*').write.csv('df_pysparked', mode='overwrite')

In [68]:
# Используя SparkSession.read(), мы можем считать данные с диска и убедиться, что они выглядят так же, как сохраненный нами DataFrame.

df_r = spark.read.csv('/Users/user/Full_ML_Course/27-PySpark/df_pysparked')
df_r.printSchema()
df_r.show(5)

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)

+-----+----+----+----+---+
|  _c0| _c1| _c2| _c3|_c4|
+-----+----+----+----+---+
|230.1|37.8|69.2|22.1|  1|
| 44.5|39.3|45.1|10.4|  2|
| 17.2|45.9|69.3| 9.3|  1|
|151.5|41.3|58.5|18.5|  1|
|180.8|10.8|58.4|12.9|  2|
+-----+----+----+----+---+
only showing top 5 rows


Похоже, этот файл не сохранил информацию о заголовках столбцов или типах данных. К сожалению, CSV никак не может сохранить информацию о своем формате. Каждый раз, когда мы его читаем, нам нужно сообщать Spark, как именно он должен быть обработан.

К счастью, существует формат файлов под названием «Parquet», который специально разработан для больших данных и решает эту проблему наряду со многими другими. Parquet обеспечивает эффективное сжатие данных, быстрее выполняет анализ, чем CSV, и сохраняет информацию о схеме набора данных. Давайте попробуем сохранить и перечитать этот файл в Parquet и обратно.

In [71]:
df.write.parquet('df_pysparked_parquet', mode='overwrite')
df_p = spark.read.parquet('/Users/user/Full_ML_Course/27-PySpark/df_pysparked_parquet')
df_p.printSchema()
df_p.show(5)

root
 |-- TV: double (nullable = true)
 |-- radio: double (nullable = true)
 |-- newspaper: double (nullable = true)
 |-- sales: double (nullable = true)
 |-- rand_col: integer (nullable = true)

+-----+-----+---------+-----+--------+
|   TV|radio|newspaper|sales|rand_col|
+-----+-----+---------+-----+--------+
|230.1| 37.8|     69.2| 22.1|       1|
| 44.5| 39.3|     45.1| 10.4|       2|
| 17.2| 45.9|     69.3|  9.3|       1|
|151.5| 41.3|     58.5| 18.5|       1|
|180.8| 10.8|     58.4| 12.9|       2|
+-----+-----+---------+-----+--------+
only showing top 5 rows


In [72]:
spark.stop()